# 1. Импорты

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

plt.style.use("ggplot")
pd.set_option("display.max_columns", 100)

# 2. Загрузка данных

In [ ]:
events = pd.read_csv("data/raw/events.csv")
category_tree = pd.read_csv("data/raw/category_tree.csv")
item_props = pd.read_csv("data/raw/item_properties.csv")

# 3. Общая информация

In [ ]:
print("Events shape:", events.shape)
print("Category tree shape:", category_tree.shape)
print("Item properties shape:", item_props.shape)

events.head()

# 4. Типы данных и пропуски

In [ ]:
events.info()

events.isnull().sum()

Вывод (написать текстом):

есть ли пропуски

где они

насколько критично

# 5. Уникальные значения

In [ ]:
print("Users:", events["visitorid"].nunique())
print("Items:", events["itemid"].nunique())
print("Events:", events.shape[0])

# 6. Распределение типов событий

In [ ]:
events["event"].value_counts()

In [ ]:
sns.countplot(data=events, x="event")
plt.title("Event distribution")
plt.show()

In [ ]:
Вывод:
обычно view >> addtocart >> transaction
сильный дисбаланс

# 7. Временной анализ

In [ ]:
events["datetime"] = pd.to_datetime(events["timestamp"], unit="ms")

events["date"] = events["datetime"].dt.date
events["hour"] = events["datetime"].dt.hour

## События по времени

In [ ]:
events.groupby("date").size().plot(figsize=(12,4), title="Events over time")
plt.show()

## По часам

In [ ]:
sns.countplot(data=events, x="hour")
plt.title("Events by hour")
plt.show()

Вывод:
пики активности
есть ли сезонность

# 8. Популярные товары

In [ ]:
top_items = events["itemid"].value_counts().head(20)

top_items.plot(kind="bar", figsize=(10,4), title="Top items")
plt.show()

Вывод:
есть сильный popularity bias

# 9. Активность пользователей

In [ ]:
user_activity = events.groupby("visitorid").size()

user_activity.describe()

In [ ]:
sns.histplot(user_activity, bins=50, log_scale=True)
plt.title("User activity distribution")
plt.show()

In [ ]:
Вывод:
большинство пользователей малоактивны
длинный хвост

# 10. Активность товаров

In [ ]:
item_activity = events.groupby("itemid").size()

sns.histplot(item_activity, bins=50, log_scale=True)
plt.title("Item popularity distribution")
plt.show()

In [ ]:
Вывод:
few popular items, many rare

# 11. Конверсия

In [ ]:
events["event"].value_counts(normalize=True)

In [ ]:
Вывод:
низкая конверсия в покупку
add-to-cart — разумный target

# 12. Разреженность (sparsity)

In [ ]:
n_users = events["visitorid"].nunique()
n_items = events["itemid"].nunique()
n_interactions = len(events)

sparsity = 1 - n_interactions / (n_users * n_items)

print("Sparsity:", sparsity)

In [ ]:
Вывод:
обычно ~99%+
значит нужен CF/ALS

# 13. Cold start

In [ ]:
user_counts = events["visitorid"].value_counts()
cold_users = (user_counts == 1).mean()

print("Cold users ratio:", cold_users)

In [ ]:
Вывод:
много новых пользователей
нужен popular fallback

# 14. Анализ item_properties

In [ ]:
item_props["property"].value_counts().head(10)

In [ ]:
item_props["value"].nunique()

In [ ]:
Вывод:
данные разреженные
можно использовать как фичи

# Финальные выводы

In [ ]:
1. Данные имеют сильный дисбаланс: большинство событий — просмотры.
2. Пользовательская активность распределена неравномерно (long tail).
3. Наблюдается сильный popularity bias среди товаров.
4. Матрица user-item сильно разрежена (>99%).
5. Высокая доля cold users → нужен fallback (Top Popular).
6. Добавление в корзину выбрано как целевое действие.
7. Подход к решению:
   - candidate generation (ALS, popularity)
   - ranking (LightGBM)